In [0]:
## 1.4: OBT (One Big Table)


In [0]:
%sql
DROP TABLE IF EXISTS propiedades.gold.obt_propiedades_completa;

CREATE TABLE propiedades.gold.obt_propiedades_completa
(
    propiedad_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
    precio DECIMAL(15,2),
    moneda STRING,
    expensas DECIMAL(15,2),
    ambientes INT,
    metros_cuadrados_totales DECIMAL(15,2),
    metros_cuadrados_cubiertos DECIMAL(15,2),
    antiguedad INT,
    cochera BOOLEAN,
    precio_por_m2 DECIMAL(15,2),
    partido STRING,
    region STRING,
    ciudad STRING,
    provincia STRING DEFAULT 'Buenos Aires',
    pais STRING DEFAULT 'Argentina',
    tipo_operacion STRING,
    categoria_operacion STRING,
    estado STRING,
    categoria_estado STRING,
    orientacion STRING,
    tipo_orientacion STRING,
    fecha_publicacion DATE,
    anio INT,
    mes INT,
    trimestre INT,
    dia_semana STRING,
    es_fin_de_semana BOOLEAN,
    segmento_precio STRING,
    rango_metros STRING,
    rango_ambientes STRING,
    tiene_expensas BOOLEAN,
    ratio_m2_cubiertos_totales DECIMAL(5,2),
    url STRING,
    _refresh_timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
)
USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported')
COMMENT 'One Big Table - Propiedades completas denormalizadas';


In [0]:
%sql

INSERT INTO propiedades.gold.obt_propiedades_completa 
(
    precio, moneda, expensas, ambientes,
    metros_cuadrados_totales, metros_cuadrados_cubiertos,
    antiguedad, cochera, precio_por_m2,
    partido, region, ciudad, provincia, pais,
    tipo_operacion, categoria_operacion,
    estado, categoria_estado,
    orientacion, tipo_orientacion,
    fecha_publicacion, anio, mes, trimestre, dia_semana, es_fin_de_semana,
    segmento_precio, rango_metros, rango_ambientes,
    tiene_expensas, ratio_m2_cubiertos_totales,
    url
)
SELECT 
    sp.precio, sp.moneda, sp.expensas, sp.ambientes,
    sp.metros_cuadrados_totales, sp.metros_cuadrados_cubiertos,
    sp.antiguedad, sp.cochera, sp.precio_por_m2,
    sp.partido,
    sp.region,
    CASE WHEN sp.region = 'capital federal' THEN 'CABA' ELSE 'GBA' END as ciudad,
    'Buenos Aires' as provincia, 'Argentina' as pais,
    sp.tipo_operacion,
    CASE 
        WHEN sp.tipo_operacion = 'alquiler' THEN 'residencial'
        WHEN sp.tipo_operacion = 'venta' THEN 'residencial'
        WHEN sp.tipo_operacion = 'alquiler_temporario' THEN 'temporal'
        ELSE 'otro'
    END as categoria_operacion,
    sp.estado,
    CASE 
        WHEN LOWER(sp.estado) LIKE '%nuevo%' THEN 'nuevo'
        WHEN LOWER(sp.estado) LIKE '%usado%' THEN 'usado'
        WHEN LOWER(sp.estado) LIKE '%remodelado%' THEN 'remodelado'
        ELSE 'sin especificar'
    END as categoria_estado,
    COALESCE(sp.orientacion, 'sin especificar') as orientacion,
    CASE 
        WHEN LOWER(sp.orientacion) LIKE '%norte%' THEN 'norte'
        WHEN LOWER(sp.orientacion) LIKE '%sur%' THEN 'sur'
        WHEN LOWER(sp.orientacion) LIKE '%este%' THEN 'este'
        WHEN LOWER(sp.orientacion) LIKE '%oeste%' THEN 'oeste'
        ELSE 'sin especificar'
    END as tipo_orientacion,
    sp.fecha_publicacion,
    YEAR(sp.fecha_publicacion) as anio, MONTH(sp.fecha_publicacion) as mes,
    QUARTER(sp.fecha_publicacion) as trimestre,
    CASE DAYOFWEEK(sp.fecha_publicacion)
        WHEN 1 THEN 'Domingo' WHEN 2 THEN 'Lunes' WHEN 3 THEN 'Martes'
        WHEN 4 THEN 'Miércoles' WHEN 5 THEN 'Jueves' WHEN 6 THEN 'Viernes'
        WHEN 7 THEN 'Sábado'
    END as dia_semana,
    DAYOFWEEK(sp.fecha_publicacion) IN (1, 7) as es_fin_de_semana,
    CASE 
        WHEN sp.moneda = 'ARS' AND sp.precio < 300000 THEN 'bajo'
        WHEN sp.moneda = 'ARS' AND sp.precio < 500000 THEN 'medio'
        WHEN sp.moneda = 'ARS' AND sp.precio < 800000 THEN 'alto'
        WHEN sp.moneda = 'ARS' THEN 'premium'
        WHEN sp.moneda = 'USD' AND sp.precio < 500 THEN 'bajo'
        WHEN sp.moneda = 'USD' AND sp.precio < 1000 THEN 'medio'
        WHEN sp.moneda = 'USD' AND sp.precio < 2000 THEN 'alto'
        WHEN sp.moneda = 'USD' THEN 'premium'
        ELSE 'sin clasificar'
    END as segmento_precio,
    CASE 
        WHEN sp.metros_cuadrados_totales < 50 THEN 'pequeño'
        WHEN sp.metros_cuadrados_totales < 100 THEN 'mediano'
        WHEN sp.metros_cuadrados_totales < 200 THEN 'grande'
        ELSE 'muy grande'
    END as rango_metros,
    CASE 
        WHEN sp.ambientes = 1 THEN 'mono'
        WHEN sp.ambientes BETWEEN 2 AND 3 THEN '2-3'
        WHEN sp.ambientes >= 4 THEN '4+'
        ELSE 'sin especificar'
    END as rango_ambientes,
    sp.expensas IS NOT NULL AND sp.expensas > 0 as tiene_expensas,
    ROUND(sp.metros_cuadrados_cubiertos / NULLIF(sp.metros_cuadrados_totales, 0), 2) as ratio_m2_cubiertos_totales,
    sp.url

FROM propiedades.silver.propiedades sp
LIMIT 10000;



In [0]:
%sql
SELECT COUNT(*) AS total_obt FROM propiedades.gold.obt_propiedades_completa;